In [ ]:
import os
import sys
sys.path.append("../")
sys.path.append("../..")
sys.path.append("../../src")
import warnings
warnings.filterwarnings("ignore")
import pickle
import torch
import numpy as np
import pandas as pd
from glob import glob
from causaldag import igsp
from causaldag import MemoizedCI_Tester, partial_correlation_test, partial_correlation_suffstat
from causaldag import MemoizedInvarianceTester, gauss_invariance_test, gauss_invariance_suffstat
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
alpha = 1e-3
alpha_inv = 1e-3

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    
    try:
        obs_suffstat = partial_correlation_suffstat(data_obs)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

        invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

        # Run IGSP

        know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
        know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
alpha = 1e-3
alpha_inv = 1e-3

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    
    try:
        obs_suffstat = partial_correlation_suffstat(data_obs)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

        invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

        # Run IGSP

        know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
        know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_2'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
alpha = 1e-3
alpha_inv = 1e-3

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    
    try:
        obs_suffstat = partial_correlation_suffstat(data_obs)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

        invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

        # Run IGSP

        know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
        know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_5'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
alpha = 1e-3
alpha_inv = 1e-3

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    
    try:
        obs_suffstat = partial_correlation_suffstat(data_obs)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

        invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

        # Run IGSP

        know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
        know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_8'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
alpha = 1e-3
alpha_inv = 1e-3

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    
    try:
        obs_suffstat = partial_correlation_suffstat(data_obs)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

        invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

        # Run IGSP

        know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
        know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]

alpha = 1e-3
alpha_inv = 1e-3


for exp_name in  ['exp_3_1', 'exp_3_2', 'exp_3_3', 'exp_3_4', 'exp_3_5']:
    os.makedirs(f'../../baselines/exps_of_result/igsp/{exp_name}', exist_ok=True)
    for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
        if idx!=5:
            continue
        print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
        if exp_name == 'exp_3_1':
            intervention_size = 0
        elif exp_name == 'exp_3_2':
            intervention_size = 8
        elif exp_name == 'exp_3_3':
            intervention_size = 12
        elif exp_name == 'exp_3_4':
            intervention_size = 16
        elif exp_name == 'exp_3_5':
            intervention_size = 20
        # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
        aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
        raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
        int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
        targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
        
        
        know_pred_graph_path = f'../../baselines/exps_of_result/igsp/{exp_name}/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        data_obs = np.array(data_list[0], dtype=np.int32)
        data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
        intervention_targets = know_targets_list[1:]
        all_nodes = [idx for idx in range(data_obs.shape[1])]
        
        try:
            obs_suffstat = partial_correlation_suffstat(data_obs)
            ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=alpha)

            invariance_suffstat = gauss_invariance_suffstat(data_obs, data_int)
            invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=alpha_inv)  

            # Run IGSP

            know_setting_list = [dict(interventions=targets) for targets in intervention_targets]
            know_est_dag = igsp(know_setting_list, all_nodes, ci_tester, invariance_tester)
            pred_I_CPDAG_know = know_est_dag.to_amat()[0].astype(np.uint8)

            with open(know_pred_graph_path, 'wb') as f:
                np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
            
            pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
            mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
            mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
            print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
        except:
            print(f'pass {benchmark_name}\n')